# Anime Voice Training Pipeline — Colab Notebook (T4)

This notebook clones the project repository, downloads required models, reads data from Google Drive, and runs the full pipeline end-to-end.

## Before you start
1. Set the runtime to **GPU** (`Runtime` -> `Change runtime type` -> `T4 GPU`).
2. Place your data under `MyDrive/UniSE/data` with the following structure:
   - `animations/` — animation episodes to process (MP4/MP3/WAV)
   - `oped/` — opening/ending audio to remove (optional)
   - `reference/` — target speaker reference audio (MP3/WAV)
3. (Recommended) Upload your UniSE checkpoint to `MyDrive/UniSE/models/unise.ckpt` as a fallback. The notebook will first try HuggingFace; if that fails it uses this local copy.
4. Get an Aliyun DashScope API key if you want Stage 2 ASR + SRT cleaning.

In [ ]:
# ============ User Configuration ============
GITHUB_REPO_URL = "https://github.com/Sufumo/Animation_audio_extractor.git"
WORK_DIR = "/content/anime_voice_training"
# Input data in Google Drive
DRIVE_DATA_DIR = "/content/drive/MyDrive/UniSE/data"
# Output results back to Google Drive (recommended for large audio files)
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/UniSE/output"
# Aliyun DashScope API key (needed for Stage 2 ASR + SRT cleaning)
DASHSCOPE_API_KEY = ""
# ============================================

In [ ]:
import os
from pathlib import Path
import shutil

# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

os.makedirs(WORK_DIR, exist_ok=True)
%cd {WORK_DIR}

# Clone / update the project
PROJECT_DIR = Path(WORK_DIR) / "repo"
if PROJECT_DIR.exists():
    !cd {PROJECT_DIR} && git pull
else:
    !git clone {GITHUB_REPO_URL} {PROJECT_DIR}

%cd {PROJECT_DIR}
print("Project ready at:", PROJECT_DIR)

## 2. Install core dependencies

Colab T4 already ships with compatible `torch` and `torchaudio`, so we skip reinstalling them.

In [ ]:
%cd {PROJECT_DIR}

# Install project requirements but skip torch/torchaudio (pre-installed on T4).
!grep -v -E "^(torch|torchaudio)" requirements.txt > /tmp/requirements_no_torch.txt
!pip install -q -r /tmp/requirements_no_torch.txt

print("Core dependencies installed.")

## 3. Clone external tool repositories

In [ ]:
%cd {WORK_DIR}

EXTERNAL_UNISE = Path(WORK_DIR) / "unified-audio" / "QuarkAudio-UniSE"
MELBAND_DIR = Path(WORK_DIR) / "Mel-Band-Roformer-Vocal-Model"

# UniSE
if EXTERNAL_UNISE.exists():
    !cd {EXTERNAL_UNISE} && git pull
else:
    !git clone https://github.com/alibaba/unified-audio.git

# Mel-Band-Roformer
if MELBAND_DIR.exists():
    !cd {MELBAND_DIR} && git pull
else:
    !git clone https://github.com/KimberleyJensen/Mel-Band-Roformer-Vocal-Model.git {MELBAND_DIR}

print("External repos ready.")

## 4. Install tool-specific dependencies

In [ ]:
# UniSE minimal inference dependencies (aligned with working UniSE_Colab.ipynb).
# Do NOT install the full requirements.txt; it contains many training-only packages
# that break on Colab (emo2vec-versa, myshell-openvoice, etc.).
!pip install -q transformers==4.46.3
!pip install -q soundfile librosa einx x-transformers pyyaml scipy numpy
!pip install -q pytorch-lightning==2.4.0

# Additional deps that may be pulled in transitively but we pin for safety.
!pip install -q einops omegaconf safetensors packaging

# Mel-Band-Roformer requirements (skip torch).
!grep -v -E "^(torch|torchaudio)" {MELBAND_DIR}/requirements.txt > /tmp/melband_requirements_no_torch.txt
!pip install -q -r /tmp/melband_requirements_no_torch.txt

# Verify critical imports.
import importlib, sys
missing = []
for pkg in ["einx", "einops", "omegaconf", "x_transformers", "safetensors", "packaging"]:
    try:
        importlib.import_module(pkg)
    except Exception as e:
        missing.append((pkg, e))
if missing:
    print("Missing packages:")
    for pkg, err in missing:
        print(f"  {pkg}: {err}")
    sys.exit(1)
else:
    print("All critical imports verified.")

## 5. Download UniSE + BiCodec checkpoints

In [ ]:
from huggingface_hub import hf_hub_download

CKPT_DIR = EXTERNAL_UNISE / "checkpoints"
os.makedirs(CKPT_DIR / "BiCodec", exist_ok=True)
os.makedirs(CKPT_DIR / "wav2vec2-large-xlsr-53", exist_ok=True)

spark_files = [
    "config.yaml",
    "BiCodec/config.yaml",
    "BiCodec/model.safetensors",
    "wav2vec2-large-xlsr-53/README.md",
    "wav2vec2-large-xlsr-53/config.json",
    "wav2vec2-large-xlsr-53/preprocessor_config.json",
    "wav2vec2-large-xlsr-53/pytorch_model.bin",
]
for f in spark_files:
    print("Downloading", f)
    hf_hub_download(repo_id="SparkAudio/Spark-TTS-0.5B", filename=f, local_dir=str(CKPT_DIR))

unise_stable = CKPT_DIR / "unise.ckpt"

# HuggingFace download sometimes fails for the original filename; try it first.
try:
    print("Downloading UniSE checkpoint from HuggingFace...")
    hf_hub_download(
        repo_id="QuarkAudio/QuarkAudio-UniSE",
        filename="epoch=20-step-109367.ckpt",
        local_dir=str(CKPT_DIR),
    )
    if not unise_stable.exists():
        shutil.copy(CKPT_DIR / "epoch=20-step-109367.ckpt", unise_stable)
except Exception as e:
    print("HuggingFace download failed:", e)
    # Fallback: user-provided checkpoint in Google Drive.
    drive_models_dir = Path("/content/drive/MyDrive/UniSE/models")
    drive_ckpt = drive_models_dir / "unise.ckpt"
    if drive_ckpt.exists():
        print("Using UniSE checkpoint from Google Drive:", drive_ckpt)
        shutil.copy(str(drive_ckpt), str(unise_stable))
    else:
        raise RuntimeError(
            f"Could not download UniSE checkpoint and no fallback found at {drive_ckpt}. "
            "Please upload your local unise.ckpt to Google Drive at MyDrive/UniSE/models/unise.ckpt "
            "and re-run this cell."
        )

print("UniSE ready. Checkpoint:", unise_stable)

## 6. Download Mel-Band-Roformer Vocal Model checkpoint

In [ ]:
MELBAND_CKPT = MELBAND_DIR / "MelBandRoformer.ckpt"
if not MELBAND_CKPT.exists():
    hf_hub_download(
        repo_id="KimberleyJSN/melbandroformer",
        filename="MelBandRoformer.ckpt",
        local_dir=str(MELBAND_DIR)
    )
print("Mel-Band-Roformer ready:", MELBAND_CKPT)

## 7. Prepare input data

Data is read directly from Google Drive at `MyDrive/UniSE/data`.

In [ ]:
DATA_DIR = Path(DRIVE_DATA_DIR)
SOURCE_DIR = DATA_DIR / "animations"
OPED_DIR = DATA_DIR / "oped"
REFERENCE_DIR = DATA_DIR / "reference"

print("Using data from:", DATA_DIR)
print("animations exists:", SOURCE_DIR.exists(), "| files:", list(SOURCE_DIR.iterdir()) if SOURCE_DIR.exists() else [])
print("oped exists:", OPED_DIR.exists(), "| files:", list(OPED_DIR.iterdir()) if OPED_DIR.exists() else [])
print("reference exists:", REFERENCE_DIR.exists(), "| files:", list(REFERENCE_DIR.iterdir()) if REFERENCE_DIR.exists() else [])

TASK_DIR = Path(WORK_DIR) / "task"

## 8. Write Colab config

In [ ]:
import yaml

DEFAULT_CONFIG = PROJECT_DIR / "configs" / "default.yaml"
COLAB_CONFIG = PROJECT_DIR / "configs" / "colab.yaml"

with open(DEFAULT_CONFIG, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

config.update({
    "source_dir": str(SOURCE_DIR),
    "oped_dir": str(OPED_DIR) if OPED_DIR.exists() else None,
    "reference_dir": str(REFERENCE_DIR),
    "task_dir": str(TASK_DIR),
    "melband_roformer": {
        "project_dir": str(MELBAND_DIR),
        "model_path": str(MELBAND_CKPT),
        "config_path": None,
    },
    "unise": {
        "project_dir": str(EXTERNAL_UNISE),
        "ckpt_path": str(CKPT_DIR / "unise.ckpt"),
        "accelerator": "auto",
        "devices": 1,
    },
    "aliyun": {
        "dashscope_api_key": DASHSCOPE_API_KEY or None,
    },
})

with open(COLAB_CONFIG, "w", encoding="utf-8") as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print("Colab config written to:", COLAB_CONFIG)

## 9. Run the pipeline

In [ ]:
import os
os.environ["DASHSCOPE_API_KEY"] = DASHSCOPE_API_KEY

%cd {PROJECT_DIR}
print("Running pipeline...")

!python {PROJECT_DIR}/main.py \
    --config {COLAB_CONFIG} \
    --task-dir {TASK_DIR} \
    --source-dir {SOURCE_DIR} \
    --reference-dir {REFERENCE_DIR} \
    {"--oped-dir" if OPED_DIR.exists() else ""} {OPED_DIR if OPED_DIR.exists() else ""} \
    --stage all \
    --resume

## 10. Download results

In [ ]:
import shutil
from pathlib import Path

RESULT_DIR = TASK_DIR / "output" / "stage2"
DRIVE_RESULT = Path(DRIVE_OUTPUT_DIR)

if RESULT_DIR.exists():
    # Copy results to Google Drive
    DRIVE_RESULT.mkdir(parents=True, exist_ok=True)
    for src in RESULT_DIR.rglob("*"):
        if src.is_file():
            dst = DRIVE_RESULT / src.relative_to(RESULT_DIR)
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(str(src), str(dst))
    print("Results copied to Google Drive:", DRIVE_RESULT)
    print("Files:", list(DRIVE_RESULT.rglob("*")))
else:
    print("No Stage 2 output found. Check the task directory:", TASK_DIR)

# Optional: also create a local zip if you want to download manually.
# from google.colab import files
# import zipfile
# ZIP_PATH = TASK_DIR / "results.zip"
# with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
#     for root, dirs, files_ in os.walk(RESULT_DIR):
#         for f in files_:
#             fp = Path(root) / f
#             zf.write(fp, arcname=fp.relative_to(TASK_DIR))
# files.download(ZIP_PATH)